In [36]:
import pandas as pd

jobs = pd.read_csv("dataset.csv")

In [37]:
jobs = jobs[jobs['job_title'].str.contains("Engineer|Scientist", case=False, na=False)]

In [38]:
print(jobs.head())


                     country country_code date_added has_expired  \
3   United States of America           US        NaN          No   
6   United States of America           US        NaN          No   
15  United States of America           US        NaN          No   
16  United States of America           US        NaN          No   
22  United States of America           US        NaN          No   

           job_board                                    job_description  \
3   jobs.monster.com  Why Join Altec? If you’re considering a career...   
6   jobs.monster.com  Report this job About the Job Based in San Fra...   
15  jobs.monster.com  CenturyLink (NYSE: CTL) is a global communicat...   
16  jobs.monster.com  Airfields. Roads. Dams. Buildings. Name the pr...   
22  jobs.monster.com  We're looking for team players to provide cons...   

                                            job_title             job_type  \
3                     Engineer - Quality Job in Dixon         

In [39]:
job_text = jobs['job_description'].iloc[0]
print(job_text[:300])

Why Join Altec? If you’re considering a career with Altec, Inc., there’s never been a better time to join us! Our Company was founded based upon values that place the customer first, and view people as our greatest strength. Altec has continued to grow, utilizing the same basic values, and over the 


In [40]:

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = ENGLISH_STOP_WORDS

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [41]:
job_clean = clean_text(job_text)

In [42]:
resumes = pd.read_csv("Resume.csv")   # adjust path if needed
print(resumes.columns)
resumes.head()

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [43]:
resumes['clean'] = resumes['Resume_str'].apply(clean_text)

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(resumes['clean'].tolist() + [job_clean])

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

resume_vectors = X[:-1]
job_vector = X[-1]

scores = cosine_similarity(resume_vectors, job_vector)

resumes['score'] = scores.flatten()   

In [46]:
skills = [
    "python", "machine", "learning", "data", "analysis", "sql",
    "excel", "nlp", "deep", "statistics",
    "communication", "project", "management",
    "engineering", "automation", "testing"
]

In [47]:
def extract_skills(text):
    found = []
    for skill in skills:
        if skill in text.lower():
            found.append(skill)
    return found

In [48]:
resumes['skills'] = resumes['clean'].apply(extract_skills)

job_skills = ["python", "machine", "learning", "sql", "data"]

def skill_gap(candidate_skills, job_skills):
    return list(set(job_skills) - set(candidate_skills))

resumes['missing_skills'] = resumes['skills'].apply(
    lambda x: skill_gap(x, job_skills)
)


print(resumes[['skills', 'missing_skills']].iloc[0])

skills            [data, analysis, statistics, management]
missing_skills            [python, learning, machine, sql]
Name: 0, dtype: object


In [49]:
def skill_match_score(candidate_skills, job_skills):
    if len(job_skills) == 0:
        return 0
    return len(set(candidate_skills) & set(job_skills)) / len(job_skills)

In [50]:
resumes['skill_score'] = resumes['skills'].apply(
    lambda x: skill_match_score(x, job_skills)
)

In [51]:
resumes['final_score'] = 0.7 * resumes['score'] + 0.3 * resumes['skill_score']

resumes = resumes.sort_values(by='final_score', ascending=False)

In [52]:
resumes[['Category', 'final_score', 'skills', 'missing_skills']].head(10)

,Category,final_score,skills,missing_skills
1762,ENGINEERING,0.371948,"[python, machine, learning, data, analysis, sq...",[]
1348,AUTOMOBILE,0.371125,"[python, machine, learning, data, analysis, sq...",[]
1218,CONSULTANT,0.350334,"[python, machine, learning, data, analysis, sq...",[]
1695,ENGINEERING,0.341476,"[python, machine, data, analysis, sql, communi...",[learning]
2153,BANKING,0.335922,"[python, machine, learning, data, analysis, sq...",[]
1807,ENGINEERING,0.335442,"[machine, learning, data, analysis, communicat...","[python, sql]"
1799,ENGINEERING,0.334435,"[python, machine, data, sql, project, manageme...",[learning]
1803,ENGINEERING,0.321832,"[machine, data, analysis, sql, excel, communic...","[python, learning]"
126,DESIGNER,0.315678,"[python, learning, data, analysis, excel, stat...","[machine, sql]"
1732,ENGINEERING,0.308111,"[python, data, sql, excel, statistics, communi...","[learning, machine]"


In [53]:
resumes[['Resume_str', 'final_score']].head(3)

,Resume_str,final_score
1762,ENGINEERING AND QUALITY TECHNICIAN ...,0.371948
1348,Highlights Prog. Languages: ...,0.371125
1218,Pavithra Shetty Summary ...,0.350334


Top candidates have higher similarity scores and better skill match with the job description. Candidates with missing critical skills receive lower final scores.

In [54]:
resumes = resumes[resumes['Category'].str.contains("Engineer|Data", case=False, na=False)]

In [55]:
import pickle

pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [56]:
top = resumes.iloc[0]

print("Top Candidate:")
print("Category:", top['Category'])
print("Score:", top['final_score'])
print("Skills:", top['skills'])
print("Missing Skills:", top['missing_skills'])

Top Candidate:
Category: ENGINEERING
Score: 0.3719480021169077
Skills: ['python', 'machine', 'learning', 'data', 'analysis', 'sql', 'excel', 'statistics', 'project', 'management', 'engineering', 'testing']
Missing Skills: []




Why Some Candidates Rank Higher

Candidates rank higher if:
- Their resume text closely matches the job description
- They have more required skills
- They have fewer missing skills



Skill Gap Insight

The system highlights missing skills such as:
- Python
- SQL
- Machine Learning

This helps recruiters quickly identify:
- Strong candidates
- Candidates needing upskilling



 Tech Stack

- Python
- Pandas
- Scikit-learn
- NLP (Text Processing)
- TF-IDF + Cosine Similarity



Files Included

- model.ipynb → main project
- Resume.csv → resume dataset
- dataset.csv → job descriptions
- vectorizer.pkl → saved model
- requirements.txt



 Real-World Use Case

This system can be used by:
- HR teams
- Recruitment platforms
- HR-tech startups

to automate resume screening and improve hiring efficiency.